# Coste de los metodos de atribucion

Mide el tiempo de ejecucion de los seis metodos de atribucion aplicados al
clasificador en las dos ramas, con los mismos parametros que
`XAI/HQ_CNN_CVAE.ipynb` y sobre la misma instancia explicada.

Responde a una afirmacion de la memoria que no estaba medida: que los metodos
basados en gradiente cuestan esencialmente lo mismo en las dos ramas mientras
que los perturbativos no. Hasta ahora la unica cifra disponible era el contador
de LIME de los informes de `experiment-5`.

El andamiaje (datos, ruido, modelos y bucle de entrenamiento) es identico al de
`XAI/linear_probe.ipynb`, de modo que ambos notebooks miden los mismos modelos.

In [ ]:
import os
import struct
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import csv
from captum.attr import Saliency, IntegratedGradients, LayerGradCam, Occlusion
from lime import lime_image
from skimage.segmentation import slic
import shap

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

DATA_SEED = 42                 # datos, particion y ruido: fijos
SEED = 42                      # la semilla del estudio XAI

device = torch.device("cpu")
print("device:", device, "| DATA_SEED", DATA_SEED, "| SEED", SEED)

## Datos: identicos a los del par gemelo

In [ ]:
RAW = os.path.join(os.path.dirname(os.getcwd()), "data", "MNIST", "raw")
if not os.path.isdir(RAW):
    RAW = os.path.join(os.getcwd(), "data", "MNIST", "raw")
print("MNIST:", RAW)


def read_idx_images(path):
    with open(path, "rb") as fh:
        magic, n, rows, cols = struct.unpack(">IIII", fh.read(16))
        buf = fh.read(n * rows * cols)
    return torch.from_numpy(np.frombuffer(buf, dtype=np.uint8).reshape(n, rows, cols).copy())


def read_idx_labels(path):
    with open(path, "rb") as fh:
        magic, n = struct.unpack(">II", fh.read(8))
        buf = fh.read(n)
    return torch.from_numpy(np.frombuffer(buf, dtype=np.uint8).copy()).long()


class PlainMNIST(Dataset):
    # Equivalente a datasets.MNIST + transforms.ToTensor()
    def __init__(self, images, labels):
        self.data = images
        self.targets = labels

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        return self.data[i].to(torch.float32).div(255.0).unsqueeze(0), int(self.targets[i])


X_train = PlainMNIST(read_idx_images(os.path.join(RAW, "train-images-idx3-ubyte"))[:400],
                     read_idx_labels(os.path.join(RAW, "train-labels-idx1-ubyte"))[:400])
X_test = PlainMNIST(read_idx_images(os.path.join(RAW, "t10k-images-idx3-ubyte"))[:100],
                    read_idx_labels(os.path.join(RAW, "t10k-labels-idx1-ubyte"))[:100])

split_generator = torch.Generator()
split_generator.manual_seed(DATA_SEED)
train_subset, val_subset = random_split(X_train, [300, 100], generator=split_generator)
print("train/val/test:", len(train_subset), len(val_subset), len(X_test))

In [ ]:
def gaussian_noise(img, sigma=0.25, generator=None):
    return torch.clamp(img + torch.randn(img.shape, generator=generator, dtype=img.dtype) * sigma, 0, 1)


def salt_pepper(img, prob=0.15, generator=None):
    noisy = img.clone()
    mask = torch.rand(img.shape, generator=generator, dtype=img.dtype)
    noisy[mask < prob / 2] = 0
    noisy[mask > 1 - prob / 2] = 1
    return noisy


def speckle(img, sigma=0.35, generator=None):
    return torch.clamp(img + img * torch.randn(img.shape, generator=generator, dtype=img.dtype) * sigma, 0, 1)


def apply_mixed_noise(img, generator):
    r = torch.randint(low=0, high=3, size=(1,), generator=generator).item()
    if r == 0:
        img = speckle(gaussian_noise(img, generator=generator), generator=generator); label = 0
    elif r == 1:
        img = salt_pepper(gaussian_noise(img, generator=generator), generator=generator); label = 1
    else:
        img = speckle(salt_pepper(img, generator=generator), generator=generator); label = 2
    return img, label


class NoisyMNISTDataset(Dataset):
    def __init__(self, mnist_dataset, seed):
        g = torch.Generator(); g.manual_seed(seed)
        self.noisy_images, self.labels = [], []
        for img, _ in mnist_dataset:
            noisy_img, label = apply_mixed_noise(img, generator=g)
            self.noisy_images.append(noisy_img)
            self.labels.append(label)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.noisy_images[idx], self.labels[idx]


train_dataset = NoisyMNISTDataset(train_subset, seed=DATA_SEED)
val_dataset = NoisyMNISTDataset(val_subset, seed=DATA_SEED + 1)
test_dataset = NoisyMNISTDataset(X_test, seed=DATA_SEED + 2)

# El orden de los lotes tambien se fija: dos ejecuciones difieren solo en los pesos.
batch_generator = torch.Generator(); batch_generator.manual_seed(DATA_SEED)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=batch_generator)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("soporte por clase (test):",
      np.bincount(np.asarray(test_dataset.labels, dtype=int), minlength=3))

## Modelos y entrenamiento

In [ ]:
estimator = Estimator()
observables = [
    SparsePauliOp.from_list([("ZIII", 1.0), ("IZII", 1.0)]),
    SparsePauliOp.from_list([("ZZII", 1.0)]),
    SparsePauliOp.from_list([("IIZZ", 1.0)]),
]


def create_qnn():
    feature_map = zz_feature_map(4, entanglement="full", reps=1)
    ansatz = real_amplitudes(4, entanglement="reverse_linear", reps=1)
    qc = QuantumCircuit(4)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)
    return EstimatorQNN(circuit=qc, input_params=feature_map.parameters,
                        weight_params=ansatz.parameters, input_gradients=True,
                        estimator=estimator, observables=observables)


class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, padding=2)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 4)

    def embed(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return torch.tanh(self.fc2(x)) * np.pi


class ClassicalNet(Backbone):
    def __init__(self):
        super().__init__()
        self.feature_mixer = nn.Sequential(
            nn.Linear(4, 16), nn.Tanh(),
            nn.Linear(16, 16), nn.Tanh(),
            nn.Linear(16, 3),
        )

    def forward(self, x):
        return self.feature_mixer(self.embed(x))


class Net(Backbone):
    def __init__(self, qnn):
        super().__init__()
        self.qnn = TorchConnector(qnn)
        self.fc_final = nn.Identity()

    def forward(self, x):
        return self.fc_final(self.qnn(self.embed(x)))

def evaluate(model):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for data, target in test_loader:
            correct += (model(data).argmax(1) == target).sum().item()
            total += target.size(0)
    return correct / total


def train(model):
    """Entrena replicando el bucle de XAI/.

    El original guarda `best_model_state = model.state_dict()` SIN copiar, asi que
    esa referencia sigue los tensores vivos y el `load_state_dict` final deja el
    modelo en la ultima epoca. Se devuelve la accuracy de ese estado final, que es
    el que produjo los resultados de la memoria, y tambien la del estado de la
    mejor epoca de validacion, que es el que la metodologia dice restaurar.
    """
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    best_val, patience, counter = float("inf"), 3, 0
    true_best_state, epochs_run, stopped_early = None, 0, False

    t0 = time.time()
    for epoch in range(10):
        epochs_run += 1
        model.train()
        for data, target in train_loader:
            optimizer.zero_grad()
            criterion(model(data), target).backward()
            optimizer.step()
        model.eval()
        vl, n = 0.0, 0
        with torch.no_grad():
            for data, target in val_loader:
                vl += criterion(model(data), target).item() * data.size(0)
                n += data.size(0)
        vl /= n
        if vl < best_val:
            best_val, counter = vl, 0
            true_best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            counter += 1
            if counter >= patience:
                stopped_early = True
                break
    secs = time.time() - t0

    # Estado final: lo que realmente deja el codigo original.
    acc_final = evaluate(model)

    # Mejor epoca real: lo que la metodologia describe. Se evalua y se descarta,
    # dejando el modelo en el estado final para que la sonda mida ese.
    if true_best_state is None:
        acc_best = acc_final
    else:
        final_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(true_best_state)
        acc_best = evaluate(model)
        model.load_state_dict(final_state)

    return acc_final, acc_best, secs, epochs_run, stopped_early

## Banco de tiempos

Cada metodo se ejecuta varias veces y se reporta la **mediana**, porque en la
rama clasica los tiempos son de milisegundos. Se descarta una pasada de
calentamiento para no medir la primera construccion del circuito.

In [ ]:
# Parametros identicos a los de XAI/HQ_CNN_CVAE.ipynb, leidos de sus celdas.
IMAGE_INDEX = 0          # la misma instancia explicada en los notebooks gemelos
IG_STEPS = 50
OCC_WINDOW = (1, 4, 4)   # ventana 4x4 con paso 4: 7x7 = 49 posiciones, sin solape
OCC_STRIDES = (1, 4, 4)
LIME_SAMPLES = 500
LIME_SEGMENTS = 20
SHAP_BACKGROUND = 50

image_test, label_real = test_dataset[IMAGE_INDEX]
image = image_test.unsqueeze(0).to(device)
img_rgb = np.stack([image_test.squeeze().cpu().numpy()] * 3, axis=-1)

background = np.array([train_dataset[i][0].numpy() for i in range(SHAP_BACKGROUND)])
background_t = torch.tensor(background, dtype=torch.float32).to(device)

# Numero de pasadas hacia delante que implica cada metodo, para leer la tabla.
n_occlusion_windows = (28 // OCC_WINDOW[1]) * (28 // OCC_WINDOW[2])
print("ventanas de Occlusion:", n_occlusion_windows, "| muestras de LIME:", LIME_SAMPLES)


def bench(fn, repeats, warmup=1):
    """Mediana de `repeats` medidas, descartando `warmup` pasadas de calentamiento.

    La mediana y no la media porque en la rama clasica los tiempos son de
    milisegundos y una sola pausa del recolector de basura desplaza la media.
    """
    for _ in range(warmup):
        fn()
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


def methods_for(model):
    """Las seis invocaciones, con los mismos argumentos que el notebook original."""
    model.eval()

    def predict_lime(images):
        gray = np.mean(images, axis=-1, keepdims=True).astype(np.float32)
        tensor = torch.tensor(gray).permute(0, 3, 1, 2).to(device)
        with torch.no_grad():
            return torch.softmax(model(tensor), dim=1).cpu().numpy()

    with torch.no_grad():
        pred = int(model(image).argmax(1).item())

    segmenter = lambda x: slic(x, n_segments=LIME_SEGMENTS, compactness=5, start_label=1)
    lime_explainer = lime_image.LimeImageExplainer()
    shap_explainer = shap.GradientExplainer(model, background_t)

    return pred, [
        ("Saliency", "gradient", 1,
         lambda: Saliency(model).attribute(image, target=pred), 5),
        ("Integrated Gradients", "gradient", IG_STEPS,
         lambda: IntegratedGradients(model).attribute(image, target=pred, n_steps=IG_STEPS), 5),
        ("Grad-CAM", "activation", 1,
         lambda: LayerGradCam(model, model.conv2).attribute(image, target=pred), 5),
        ("SHAP (gradient)", "gradient", SHAP_BACKGROUND,
         lambda: shap_explainer.shap_values(image), 3),
        ("Occlusion", "perturbation", n_occlusion_windows,
         lambda: Occlusion(model).attribute(image, strides=OCC_STRIDES, target=pred,
                                            sliding_window_shapes=OCC_WINDOW,
                                            baselines=0), 3),
        ("LIME", "perturbation", LIME_SAMPLES,
         lambda: lime_explainer.explain_instance(img_rgb, predict_lime, top_labels=3,
                                                 hide_color=0, num_samples=LIME_SAMPLES,
                                                 segmentation_fn=segmenter), 3),
    ]

## Ejecucion

Entrena los dos clasificadores a la semilla 42 y cronometra los seis metodos.
El clasico tarda menos de un segundo; el hibrido, unos nueve minutos.

In [ ]:
timings = {}

for branch, build in (("classical", ClassicalNet),
                      ("hybrid", lambda: Net(create_qnn()))):
    torch.manual_seed(SEED)
    model = build().to(device)
    acc, best, secs, epochs, early = train(model)
    print("%-9s | accuracy %.2f (mejor epoca %.2f) | entrenamiento %.1f s | %d epocas%s"
          % (branch, acc, best, secs, epochs, ", early" if early else ""), flush=True)

    pred, methods = methods_for(model)
    print("           prediccion sobre la imagen explicada:", pred, "| etiqueta:", label_real,
          flush=True)
    for name, family, passes, fn, repeats in methods:
        t = bench(fn, repeats)
        timings.setdefault(name, {"family": family, "passes": passes})[branch] = t
        print("           %-22s %9.4f s" % (name, t), flush=True)
    timings.setdefault("_train", {"family": "-", "passes": 0})[branch] = secs
    print(flush=True)

## Resultados

In [ ]:
RESULTS = os.path.join(os.getcwd(), "results")
os.makedirs(RESULTS, exist_ok=True)
csv_path = os.path.join(RESULTS, "attribution_cost.csv")

rows = [(n, v["family"], v["passes"], v["classical"], v["hybrid"],
         v["hybrid"] / v["classical"])
        for n, v in timings.items() if n != "_train"]

with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["method", "family", "forward_passes", "classical_s", "hybrid_s", "ratio"])
    for r in rows:
        w.writerow([r[0], r[1], r[2], "%.6f" % r[3], "%.6f" % r[4], "%.1f" % r[5]])
print("guardado en", csv_path)
print()

print("%-22s %-13s %6s %12s %12s %8s" %
      ("metodo", "familia", "pasadas", "clasico (s)", "hibrido (s)", "razon"))
print("-" * 78)
for r in rows:
    print("%-22s %-13s %6d %12.4f %12.4f %8.1f" % r)
print()
tr = timings["_train"]
print("entrenamiento (referencia)   %12.1f %12.1f %8.1f"
      % (tr["classical"], tr["hybrid"], tr["hybrid"] / tr["classical"]))
print()
lime = [r for r in rows if r[0] == "LIME"][0]
print("LIME, muestras por segundo:  clasico %.0f  hibrido %.0f"
      % (LIME_SAMPLES / lime[3], LIME_SAMPLES / lime[4]))
print("(los informes de experiment-5 registraron 3937 y 205 it/s)")

In [ ]:
# Filas listas para pegar en la tabla del TFM.
for r in rows:
    print("\t\t%s & %s & $%d$ & $%.4f$ & $%.4f$ & $%.0f\\times$ \\\\"
          % (r[0], r[1].capitalize(), r[2], r[3], r[4], r[5]))